# Notebook 07 — End-to-End Pipeline Generation

**Phase 8 demo**: process a single real paper from PMID to a generated, executable pipeline.

Full workflow:
1. [Configuration](#1-configuration)
2. [Parse paper](#2-parse-paper)
3. [Parse figures](#3-parse-figures)
4. [Parse methods](#4-parse-methods)
5. [Fetch datasets](#5-fetch-datasets)
6. [Parse software](#6-parse-software)
7. [Build pipeline](#7-build-pipeline)
8. [Inspect outputs](#8-inspect-outputs)
9. [Export to disk](#9-export-to-disk)

---
### 🐛 Bug fixed (2026-03-31)

**Root cause**: `parse_jats_xml` correctly extracted `figure_captions` from JATS XML `<fig>` elements, but `_build_paper_from_jats` discarded them — they were only used to seed `figure_ids` and were never stored in the `Paper` model. As a result, `FigureParser._find_caption` always returned `""`, causing `_decompose_subfigures` to short-circuit and return an empty list for every figure.

**Fix** (3 files, ~20 lines):
1. **`models/paper.py`**: Added `figure_captions: dict[str, str]` field to `Paper`.
2. **`parsers/paper_parser.py`**: `_build_paper_from_jats` now passes `figure_captions` when constructing the `Paper` object.
3. **`parsers/figure_parser.py`**: `_find_caption` now has a parent-figure fallback — sub-panel IDs like `"Figure 1a"` inherit `"Figure 1"`'s caption so they still get subfigure decomposition.

Re-run from cell 3 onwards to see populated `subfigures`, `datasets_used`, and `methods_used`.

In [1]:
import json
import os
import logging
from pathlib import Path

# Optional: set NCBI_API_KEY for higher rate limits
os.environ["NCBI_API_KEY"] = "9e09d5d38c680a8358426f7fac6d154b4f08"
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-REDACTED"

logging.basicConfig(level=logging.WARNING)  # suppress INFO noise in notebook
logger = logging.getLogger("researcher_ai")
logger.setLevel(logging.INFO)

from researcher_ai.parsers.paper_parser import PaperParser
from researcher_ai.parsers.figure_parser import FigureParser
from researcher_ai.parsers.methods_parser import MethodsParser
from researcher_ai.parsers.data.geo_parser import GEOParser
from researcher_ai.parsers.data.sra_parser import SRAParser
from researcher_ai.parsers.software_parser import SoftwareParser
from researcher_ai.pipeline.builder import PipelineBuilder
from researcher_ai.models.pipeline import PipelineBackend

print("researcher_ai imports OK")

researcher_ai imports OK


## 1. Configuration

In [2]:
# ── Paper source ──────────────────────────────────────────────────────────────
# Replace with any PMID, PMCID, DOI, or local PDF path.
# PMID 27018577: Van Nostrand et al. 2016 (eCLIP) — open-access, well-structured methods
paper_source = "27018577"

# ── LLM cache dir ─────────────────────────────────────────────────────────────
# Point to a directory to cache LLM responses and avoid re-billing API calls
# during iterative notebook development. Set to None to disable.
cache_dir = Path("../tests/snapshots")

# ── Pipeline backend ──────────────────────────────────────────────────────────
backend = PipelineBackend.SNAKEMAKE  # or PipelineBackend.NEXTFLOW

# ── Output directory ──────────────────────────────────────────────────────────
output_dir = Path(f"output/pmid_{paper_source}")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Paper source : {paper_source}")
print(f"Backend      : {backend.value}")
print(f"Cache dir    : {cache_dir}")
print(f"Output dir   : {output_dir}")

Paper source : 27018577
Backend      : snakemake
Cache dir    : ../tests/snapshots
Output dir   : output/pmid_27018577


## 2. Parse paper

In [3]:
parser = PaperParser(cache_dir=cache_dir)
paper = parser.parse(paper_source)

print(f"Title     : {paper.title}")
print(f"PMID      : {paper.pmid}")
print(f"PMCID     : {paper.pmcid}")
print(f"DOI       : {paper.doi}")
print(f"Type      : {paper.paper_type.value}")
print(f"Sections  : {len(paper.sections)}")
print(f"Figures   : {paper.figure_ids}")
print(f"Supp items: {len(paper.supplementary_items)}")

INFO:researcher_ai.parsers.paper_parser:Parsing paper from 27018577 (type: pmid)


Title     : Robust transcriptome-wide discovery of RNA binding protein binding sites with enhanced CLIP (eCLIP)
PMID      : 27018577
PMCID     : PMC4887338
DOI       : 10.1038/nmeth.3810
Type      : experimental
Sections  : 4
Figures   : ['Figure 1', 'Figure 2', 'Figure 3', 'Figure 4', 'Figure 1a', 'Figure 1b', 'Figure 1b–c', 'Figure 1d', 'Figure 2a', 'Figure 2b', 'Figure 2c', 'Figure 2d', 'Figure 2e', 'Figure 3a', 'Figure 3a–b', 'Figure 3b', 'Figure 3c', 'Figure 3d', 'Figure 3e', 'Figure 3f', 'Figure 4a', 'Figure 4a–d', 'Figure 4b', 'Figure 4c–d', 'Figure 5a', 'Figure 5b–c', 'Figure 5d', 'Figure 6a–f', 'Figure 7a–f', 'Figure 8a–b', 'Figure 8b', 'Figure 8c–d', 'Figure 8e', 'Figure 9a', 'Figure 9b', 'Figure 9c–d', 'Figure 10d', 'Figure 10e', 'Figure 12a', 'Figure 12b–c', 'Figure 13a', 'Figure 13b', 'Figure 13c', 'Figure 13d', 'Figure 14a-b', 'Figure 14a–c', 'Figure 14b', 'Figure 15c', 'Figure 15d–e', 'Figure 15f', 'Figure 15g', 'Supplementary Figure 1a', 'Supplementary Figure 1b', 'Supp

## 3. Parse figures

> **After the 2026-03-31 fix**, each main figure now has:
> - `subfigures` — list of panels (a, b, c…) with plot_type, axes, assays
> - `datasets_used` — GEO/SRA accessions extracted from caption text  
> - `methods_used` — assay and method names (eCLIP, CLIPper, RNA-seq…)
>
> Sub-panel IDs (e.g. `Figure 1a`) inherit the parent figure's caption via the new fallback in `_find_caption`.

In [4]:
fig_parser = FigureParser(cache_dir=cache_dir)
figures = fig_parser.parse_all_figures(paper)

print(f"Parsed {len(figures)} figures")
for fig in figures[:10]:   # show first 10 for brevity
    print(f"\n  {fig.figure_id}: {fig.title[:80]}")
    print(f"    Caption    : {repr(fig.caption[:60]) if fig.caption else '(no caption)'}")
    print(f"    Subfigures : {[sf.label for sf in fig.subfigures]}")
    print(f"    Datasets   : {fig.datasets_used}")
    print(f"    Methods    : {fig.methods_used[:4]}")

# Sanity check — at least the main numbered figures should be populated
main_figs = [f for f in figures if f.figure_id in
             ('Figure 1', 'Figure 2', 'Figure 3', 'Figure 4')]
empty_main = [f.figure_id for f in main_figs
              if not f.subfigures or not f.methods_used]
if empty_main:
    print(f"\n⚠️  These main figures still have empty subfigures/methods: {empty_main}")
    print("   → Check that paper.figure_captions is populated (requires full JATS XML fetch).")
else:
    print(f"\n✓  All {len(main_figs)} main figures have non-empty subfigures and methods.")

INFO:researcher_ai.parsers.figure_parser:Parsing Figure 1
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 2
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 3
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 4
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 1a
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 1b
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 1b–c
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 1d
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 2a
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 2b
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 2c
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 2d
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 2e
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 3a
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 3a–b
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 3b
INFO:researcher_ai.parsers.figure_parser:Parsing Figure 

Parsed 87 figures

  Figure 1: eCLIP-seq improves RBP target identification over iCLIP
    Caption    : 'Improved identification of RNA binding protein (RBP) targets'
    Subfigures : ['a', 'b', 'c', 'd']
    Datasets   : []
    Methods    : ['eCLIP-seq', 'iCLIP', 'UV crosslinking', 'RNase I digestion']

  Figure 2: SMInput Normalization Improves eCLIP Signal, Specificity, and Reproducibility
    Caption    : 'Improved CLIP signal-to-noise and reproducibility by normali'
    Subfigures : ['a', 'b', 'c', 'd', 'e']
    Datasets   : []
    Methods    : ['eCLIP', 'CLIP', 'CLIPper peak calling', 'SMInput normalization']

  Figure 3: Scalable eCLIP Enables High-Throughput RBP Target Identification Across Cell Lin
    Caption    : 'Scalable RBP target identification with eCLIP (a) Pie charts'
    Subfigures : ['a', 'b', 'c']
    Datasets   : []
    Methods    : ['eCLIP', 'iCLIP', 'CLIP', 'PAR-CLIP']

  Figure 4: eCLIP Identifies Protein Binding to Abundant Noncoding RNAs
    Caption    : 'eCL

## 4. Parse methods

In [5]:
methods_parser = MethodsParser(cache_dir=cache_dir)
method = methods_parser.parse(paper, figures)

print(f"Paper DOI : {method.paper_doi}")
print(f"Assays    : {len(method.assays)}")
print(f"Deps      : {len(method.assay_graph.dependencies)}")
print()
for assay in method.assays:
    print(f"  [{assay.name}]  {len(assay.steps)} steps")
    for step in assay.steps[:3]:  # show first 3 steps
        print(f"    Step {step.step_number}: {step.description[:70]}")
    if len(assay.steps) > 3:
        print(f"    ... and {len(assay.steps) - 3} more")

Paper DOI : 10.1038/nmeth.3810
Assays    : 7
Deps      : 1

  [eCLIP-seq library preparation]  11 steps
    Step 1: RNA-binding protein–RNA interactions are stabilized by UV crosslinking
    Step 2: Cells are lysed in iCLIP lysis buffer to release RBP-RNA complexes, fo
    Step 3: RBP-RNA complexes are immunoprecipitated using a target-specific prima
    ... and 8 more
  [eCLIP-seq data processing]  13 steps
    Step 1: Standard HiSeq demultiplexing followed by inline barcode demultiplexin
    Step 2: Adapter trimming of demultiplexed reads; reads shorter than 18 bp afte
    Step 3: Reads are first mapped against human repetitive elements in RepBase to
    ... and 10 more
  [Normalization of eCLIP signal against SMInput]  6 steps
    Step 1: SMInput samples are processed identically to eCLIP samples through the
    Step 2: Count the number of eCLIP reads overlapping CLIPper-identified peaks a
    Step 3: Calculate enrichment p-values for each peak using Yates' Chi-Square te
    ... and

## 5. Fetch datasets

In [6]:
geo = GEOParser()
sra = SRAParser()

# Collect candidate accessions from method + figure parsing
candidate_accessions: set[str] = set()
for assay in method.assays:
    if assay.raw_data_source:
        import re
        candidate_accessions.update(re.findall(r'GSE\d+|SRP\d+|ERP\d+', assay.raw_data_source))
for fig in figures:
    candidate_accessions.update(fig.datasets_used)

print(f"Candidate accessions: {sorted(candidate_accessions)}")
print()

datasets = []
resolved_accessions: list[str] = []
seen_dataset_ids: set[str] = set()

def _add_dataset(ds):
    if ds.accession in seen_dataset_ids:
        return
    seen_dataset_ids.add(ds.accession)
    datasets.append(ds)
    resolved_accessions.append(ds.accession)

for acc in sorted(candidate_accessions):
    try:
        if acc.startswith("GSE"):
            # Resolve SuperSeries one level deep so child/subseries are captured.
            ds = geo.parse(acc, recursive=True)
            _add_dataset(ds)
            for child in getattr(ds, 'related_datasets', []) or []:
                try:
                    child_ds = geo.parse(child, recursive=False)
                    _add_dataset(child_ds)
                except Exception as child_e:
                    print(f"    child {child}: not resolved — {child_e}")
        elif acc.startswith(("SRP", "ERP")):
            ds = sra.parse(acc)
            _add_dataset(ds)
        else:
            continue
    except Exception as e:
        print(f"  {acc}: not resolved — {e}")

for ds in datasets:
    print(f"  {ds.accession}: {ds.title[:60]}")
    print(f"    Organism: {ds.organism}  |  Samples: {ds.total_samples}")

print(f"\nResolved accessions: {sorted(set(resolved_accessions))}")
print(f"Total datasets: {len(datasets)}")


Accessions found: []


Total datasets: 0


## 6. Parse software

In [ ]:
sw_parser = SoftwareParser(cache_dir=cache_dir)
software = sw_parser.parse_from_method(method)

print(f"Software tools identified: {len(software)}")
for sw in software:
    pkg = sw.bioconda_package or sw.pypi_package or sw.cran_package or "—"
    print(f"  {sw.name:<20} v{sw.version or '?':<12} pkg={pkg}")

## 7. Build pipeline

In [ ]:
builder = PipelineBuilder()
pipeline = builder.build(
    method=method,
    datasets=datasets,
    software=software,
    figures=figures,
    backend=backend,
)

print("Pipeline built!")
print(f"  Name    : {pipeline.config.name}")
print(f"  Backend : {pipeline.config.backend.value}")
print(f"  nf-core : {pipeline.config.nf_core_pipeline or 'custom'}")
print(f"  Steps   : {len(pipeline.config.steps)}")

if pipeline.config.nf_core_pipeline:
    print(f"\n  ✓ nf-core/{pipeline.config.nf_core_pipeline} pipeline detected — params.yaml generated")
else:
    print("\n  ✓ Custom Snakemake pipeline generated")

print("\nExecution order:")
for step_id in pipeline.config.execution_order():
    step = next(s for s in pipeline.config.steps if s.step_id == step_id)
    deps = f"← {step.depends_on}" if step.depends_on else ""
    print(f"  {step_id:<35} {deps}")

## 8. Inspect outputs

In [ ]:
print("=== Snakefile ===")
print(pipeline.snakefile_content)

In [ ]:
print("=== environment.yml ===")
print(pipeline.conda_env_yaml)

In [ ]:
nb = json.loads(pipeline.jupyter_content)
print(f"Generated notebook: {nb['nbformat']}.{nb['nbformat_minor']} format")
print(f"Cells: {len(nb['cells'])}")
print()
for i, cell in enumerate(nb['cells'][:10]):
    src = cell['source']
    first_line = (src if isinstance(src, str) else "".join(src)).splitlines()[0][:70]
    print(f"  Cell {i:02d} [{cell['cell_type']:8s}]: {first_line}")
if len(nb['cells']) > 10:
    print(f"  ... and {len(nb['cells']) - 10} more cells")

## 9. Export to disk

In [ ]:
files_written = []

if pipeline.snakefile_content:
    p = output_dir / "Snakefile"
    p.write_text(pipeline.snakefile_content)
    files_written.append(p)

if pipeline.nextflow_content:
    p = output_dir / "params.yaml"
    p.write_text(pipeline.nextflow_content)
    files_written.append(p)

if pipeline.jupyter_content:
    p = output_dir / "figure_reproduction.ipynb"
    p.write_text(pipeline.jupyter_content)
    files_written.append(p)

if pipeline.conda_env_yaml:
    p = output_dir / "environment.yml"
    p.write_text(pipeline.conda_env_yaml)
    files_written.append(p)

# Save the full pipeline spec as JSON for inspection / re-use
pipeline_json_path = output_dir / "pipeline_spec.json"
pipeline_json_path.write_text(
    json.dumps(pipeline.model_dump(), indent=2, default=str)
)
files_written.append(pipeline_json_path)

print("Pipeline files written:")
for f in files_written:
    size = f.stat().st_size
    print(f"  {str(f):<60} ({size:,} bytes)")
print(f"\nTotal: {len(files_written)} files in {output_dir}/")